# Week 7 participant practical: repeated persistence through physical time

This is the student investigation, built around a synthetic ring that fills in and re-forms. The setup and mathematical framing are supplied. You must record predictions, complete short computational steps, check intermediate objects and justify an interpretation.

We compute ordinary Rips persistence independently at each frame of a synthetic evolving point cloud. The output is a time series of diagrams and a CROCKER-style rank surface. No feature identity is asserted across frames.

**Working rule.** Run one section at a time. Before each TODO, state what shape, dimension or direction you expect in the output. Optional extensions come only after the core checkpoints agree.



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
RNG=np.random.default_rng(3024)
from ripser import ripser

def betti_curve(D,grid): return np.array([np.sum((D[:,0]<=a)&(a<D[:,1])) for a in grid])
def max_persistence(D):
 F=D[np.isfinite(D[:,1])]; return float(np.max(F[:,1]-F[:,0])) if len(F) else 0.

## Representation bridge: delay coordinates

A scalar signal is not yet a state-space point cloud. For lag $\tau$ and dimension $m$ construct

$$\Phi_{m,\tau}(t)=\bigl(x(t),x(t-\tau),\ldots,x(t-(m-1)\tau)\bigr).$$

The code below uses index lag rather than physical units. Predict the shape for a periodic signal. Then change only `lag` and describe what geometric organisation changes. Temporal order is not retained by the later Rips calculation unless it is stored separately.

In [ ]:
def delay_coordinates(signal, dimension, lag):
    rows = len(signal) - (dimension - 1) * lag
    return np.column_stack([
        signal[(dimension - 1 - j) * lag:(dimension - 1 - j) * lag + rows]
        for j in range(dimension)
    ])

sample_times = np.linspace(0, 8 * np.pi, 500)
signal = np.sin(sample_times)
lag = 25
embedded = delay_coordinates(signal, dimension=2, lag=lag)
plt.figure(figsize=(4, 4))
plt.plot(embedded[:, 0], embedded[:, 1], linewidth=1)
plt.gca().set_aspect('equal')
plt.xlabel(r'$x(t)$'); plt.ylabel(r'$x(t-\tau)$')
plt.show()
print('embedded shape:', embedded.shape, 'index lag:', lag)

## 1. Observe: participant checkpoint

The synthetic system moves from a coherent ring to a filled cloud and back. The samples at each frame are observations of a changing representation. The point labels are reused only to generate smooth motion; persistence does not use those labels.

In [ ]:
n=70; times=np.linspace(0,1,17); theta=np.linspace(0,2*np.pi,n,endpoint=False)
base_angle=theta+.03*RNG.normal(size=n); target_radius=np.sqrt(RNG.random(n))
frames=[]
for t in times:
 mix=np.sin(np.pi*t)**2
 radius=(1-mix)*np.ones(n)+mix*target_radius
 P=np.c_[radius*np.cos(base_angle),radius*np.sin(base_angle)]+.025*RNG.normal(size=(n,2))
 frames.append(P)
fig,axes=plt.subplots(1,3,figsize=(9,3))
for ax,i in zip(axes,[0,8,16]): ax.scatter(*frames[i].T,s=12); ax.set_title(f't={times[i]:.2f}'); ax.set_aspect('equal')
plt.show()

## 2. Predict: participant checkpoint

1. When should the longest $H_1$ interval be largest?
2. At a fixed Rips threshold, when should $\beta_1$ be nonzero?
3. Does a long bar at consecutive frames establish that it is the same feature?
4. How might window length change the scientific interpretation?

## 3. Implement: participant checkpoint

Start with the three displayed frames at indices `0`, `8` and `16`. Compute their separate $H_1$ diagrams and longest persistence. Check your prediction before scaling the same operation to all frames.

Keep physical time $t$ distinct from filtration threshold $\varepsilon$.

In [ ]:
check_indices = [0, 8, 16]
for index in check_indices:
    diagram = ripser(frames[index], maxdim=1)['dgms'][1]
    print('index:', index, 'time:', times[index],
          'longest H1:', max_persistence(diagram))

# TODO: use the same expression in a list comprehension for every frame.
diagrams_by_time = []
# TODO: calculate max_persistence for every diagram and plot it against times.
maximum_by_time = np.array([])

## 4. Compare: participant checkpoint

Build the CROCKER-style array in two stages:

1. choose `grid = np.linspace(0, 1.8, 150)` and test `betti_curve` on the first diagram;
2. stack one curve per physical time, expecting shape `(17, 150)`.

Then compute radial coefficient of variation as a simpler baseline. The topology and baseline must use the same frames.

In [ ]:
grid = np.linspace(0, 1.8, 150)

# TODO: after diagrams_by_time is complete, calculate the first Betti curve.
first_curve = np.array([])
print('expected first-curve length:', len(grid), 'actual:', len(first_curve))

# TODO: stack all curves. Check that B.shape == (len(times), len(grid)).
B = np.empty((0, len(grid)))

def radial_coefficient_of_variation(points):
    radii = np.linalg.norm(points - points.mean(axis=0), axis=1)
    return radii.std() / radii.mean()

# TODO: compute radial_cv for every frame.
radial_cv = np.array([])
# TODO: display B.T above radial_cv using a shared physical-time axis.

## 5. Interpret: participant checkpoint

1. Which axis is physical time and which is filtration scale?
2. What statement can the rank surface support without feature tracking?
3. When would a sliding window blur a transition or create apparent persistence?
4. Does topology add information beyond radial variation in this synthetic example?
5. What extra maps or correspondences would be required to claim feature identity?

**† Qualification.** A smooth time series of summaries is still repeated static persistence. It is not a vineyard or vineyard module.